In [1]:
!pip install -q google-genai

In [2]:
from google import genai
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GOOGLE_API_KEY = secrets.get_secret("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)
print("✅ Connected to Gemma successfully!")


✅ Connected to Gemma successfully!


In [3]:
system_prompt = """You are Aarogya, a friendly rural health assistant helping people 
in Indian villages who don't have easy access to doctors.

CRITICAL EMERGENCY SIGNS - Always mark these as EMERGENCY immediately:
- Chest pain with arm/jaw/back pain or numbness
- Difficulty breathing or shortness of breath
- Signs of stroke: face drooping, arm weakness, speech difficulty
- Severe bleeding that won't stop
- Loss of consciousness or seizures
- High fever in infants under 3 months
- Severe head injury
- Suspected poisoning

When someone describes their symptoms:
1. Start with: EMERGENCY / MODERATE / MILD on its own line
2. Explain in 2 simple sentences what might be happening
3. Give 2-3 home remedies if MILD or MODERATE
4. If EMERGENCY: say GO TO HOSPITAL IMMEDIATELY in bold, no home remedies
5. Respond in the same language the patient uses
6. Always end with: 'This is not a substitute for professional medical advice.'

Be warm, caring and simple. No medical jargon.
When in doubt between MODERATE and EMERGENCY, always choose EMERGENCY."""

In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("""
<div style="font-family: Arial; max-width: 600px; margin: 20px auto; 
            background: #f0f7f0; padding: 20px; border-radius: 15px;">
    <h2 style="color: #2e7d32; text-align: center;">🏥 Aarogya - Village Health Voice</h2>
    <p style="text-align: center; color: #555;">AI Health Assistant for Rural Communities</p>
</div>
"""))

symptom_input = widgets.Textarea(
    placeholder='Describe your symptoms here... (English or Tamil)',
    layout=widgets.Layout(width='600px', height='100px')
)

language_select = widgets.Dropdown(
    options=['Auto Detect', 'English', 'Tamil', 'Hindi', 'Telugu'],
    value='Auto Detect',
    description='Language:',
)

submit_button = widgets.Button(
    description='Check Symptoms',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output()

def on_submit(b):
    with output_area:
        output_area.clear_output()
        text = symptom_input.value.strip()
        if not text:
            print("Please describe your symptoms first!")
            return
        
        lang = language_select.value
        lang_instruction = ""
        if lang != "Auto Detect":
            lang_instruction = f" Please respond in {lang}."
        
        print("🔍 Analyzing your symptoms...")
        result = check_symptoms(text + lang_instruction)
        
        # Color code based on severity
        if "EMERGENCY" in result:
            color = "#ff4444"
            emoji = "🚨"
        elif "MODERATE" in result:
            color = "#ff8800"
            emoji = "⚠️"
        else:
            color = "#44aa44"
            emoji = "✅"
            
        display(HTML(f"""
        <div style="background: white; padding: 15px; border-radius: 10px; 
                    border-left: 5px solid {color}; margin-top: 10px;">
            <h3 style="color: {color};">{emoji} Aarogya's Assessment</h3>
            <p style="white-space: pre-wrap; color: #333;">{result}</p>
        </div>
        """))

submit_button.on_click(on_submit)

display(widgets.VBox([
    widgets.HTML("<b>Describe your symptoms:</b>"),
    symptom_input,
    language_select,
    submit_button,
    output_area
]))